In [1]:
import os, sys, time

from requests import options
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../')))
from src.utils import *
from src.panda_program import PandaMugProgram
from src.generic_program import ProgramOptions
from pydrake.all import (
    StartMeshcat,
)
from tqdm import tqdm


ikflow/config.py | Using device: 'cuda:0'


In [2]:

####### Options #######
num_tests = 100
num_initial_guesses = 10
program_options = ProgramOptions(
    visualize=True,
    joint_centering_cost=0,
    max_wall_time=60.0,
    which_solver='ipopt',
    acceptable_tol = 1e-3,
    acceptable_constr_viol_tol = 1e-4,
    ik_constraint_tol = (1e-6, 0.01),
    mug_height = 0.04,
    vars_file = "vars_file.txt"
)

In [3]:

meshcat = StartMeshcat()

yaml_file = os.path.join(RepoDir(), "models/panda/panda_finray_collision_backup.yaml")
base_diagram = BuildEnv(meshcat=meshcat, directives_file = yaml_file)
program = PandaMugProgram(base_diagram)
program.create_prog()


ik_solver = program.ik_solver

successes = 0
times = []
costs = []

INFO:drake:Meshcat listening for connections at http://localhost:7002


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/panda_arm_hand_formatted_link_filepaths_absolute.urdf
joint mimic: no multiplier, using default value of 1 
joint mimic: no offset, using default value of 0 
URDFParser: Link size: 17
URDFParser: Joint size: 12
Geometry: Loading 12 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link0.dae into Group
ManagedGeometry: loaded /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link0.dae in time 0.208874s
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link3.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link4.dae into Group
Geometry: Loading 3 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link5.dae into Group
Geometry: Loadi

In [4]:
mug = Mug(middle = RigidTransform(p=[0.75, 0.0, 0.05]))
program = PandaMugProgram(base_diagram, options = program_options, model=ik_solver)
program.create_prog(target_mug = mug)
result = program.Solve()


program.plant.GetPositions(program.plant_context)

array([-2.17654753, -1.29292393,  2.17816615, -1.60432971, -1.0717082 ,
        2.49344993,  0.44715035])

In [5]:
meshcat2 = StartMeshcat()
diagram2 = BuildEnv(meshcat=meshcat2, directives_file = yaml_file)


INFO:drake:Meshcat listening for connections at http://localhost:7003


In [6]:
diagram_context2 = diagram2.CreateDefaultContext()
diagram2.ForcedPublish(diagram_context2)
plant2 = diagram2.GetSubsystemByName("plant")
plant_context2 = plant2.GetMyContextFromRoot(diagram_context2)

==== LCM Warning ===
LCM detected that large packets are being received, but the kernel UDP
receive buffer is very small.  The possibility of dropping packets due to
insufficient buffer space is very high.

For more information, visit:
   https://lcm-proj.github.io/lcm/content/multicast-setup.html

==== LCM Warning ===
LCM detected that large packets are being received, but the kernel UDP
receive buffer is very small.  The possibility of dropping packets due to
insufficient buffer space is very high.

For more information, visit:
   https://lcm-proj.github.io/lcm/content/multicast-setup.html



In [9]:
plant2.GetPositions(plant_context2)
three_sols = np.zeros(21)
three_sols[:7] = np.array([-2.17654753, -1.29292393,  2.17816615, -1.60432971, -1.0717082 , 2.49344993,  0.44715035])
three_sols[7:14] = np.array([-0.12958367,  1.42982352, -0.04966962, -0.30710962,  0.49052745, 1.46310174,  1.65580523])
three_sols[14:] = np.array([-0.23298982,  0.92368752,  0.09019972, -1.80501652,  2.5089488 , 1.90582228,  0.92483068])
plant2.SetPositions(plant_context2, three_sols)

opacity = 0.7
meshcat2.SetProperty(f"/drake/illustration/panda", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/finray/panda_hand", "opacity", 0.3)
meshcat2.SetProperty(f"/drake/illustration/panda2", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/finray2", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/finray2/panda_hand", "opacity", 0.3)
meshcat2.SetProperty(f"/drake/illustration/panda3", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/finray3", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/finray3/panda_hand", "opacity", 0.3)


diagram2.ForcedPublish(diagram_context2)